# sMAB Simulation

This notebook shows a simulation framework for the stochastic multi-armed bandit (sMAB). It allows to study the behaviour of the bandit algoritm, to evaluate results and to run experiments on simulated data under different reward and action settings.

In [1]:
from pybandits.model import Beta
from pybandits.smab import SmabBernoulli
from pybandits.smab_simulator import SmabSimulator

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


First we need to define the simulation parameters. The parameters contain:
- Number of update rounds
- Number of samples per batch of update round
- Seed for reproducibility
- Verbosity enabler
- Visualization enabler

Data are processed in batches of size n>=1. Per each batch of simulated samples, the sMAB selects one action and collects the corresponding simulated reward for each sample. Then, prior parameters are updated based on returned rewards from recommended actions.

In [2]:
# general simulator parameters
n_updates = 10
batch_size = 100
random_seed = None
verbose = True
visualize = True

Next, we initialize the action model and the sMAB. We define three actions, each with a Beta model. The Beta model is a conjugate prior for the Bernoulli likelihood function. The Beta distribution is defined by two parameters: alpha and beta. The action model is defined as a dictionary with the action name as key and the Beta model as value.

In [3]:
# define action model
actions = {
    "a1": Beta(),
    "a2": Beta(),
    "a3": Beta(),
}
# init stochastic Multi-Armed Bandit model
smab = SmabBernoulli(actions=actions)

Finally, we need to define the probabilities of positive rewards per each action, i.e. the ground truth ('Action A': 0.8 that if the bandits selects 'Action A' for samples that belong to group '0', then the environment will return a positive reward with 80% probability).


In [4]:
# init probability of rewards from the environment
probs_reward = dict(zip(actions.keys(), [0.05, 0.80, 0.05]))
print("Probability of positive reward for each action:")
probs_reward

Probability of positive reward for each action:


{'a1': 0.05, 'a2': 0.8, 'a3': 0.05}

Now, we initialize the SmabSimulator with the parameters set above.

In [5]:
# init simulation
smab_simulator = SmabSimulator(
    mab=smab,
    batch_size=batch_size,
    n_updates=n_updates,
    probs_reward=probs_reward,
    verbose=verbose,
    visualize=visualize,
)

Now, we can start simulation process by executing run() which performs the following steps:
```
For i=0 to n_updates:
    Extract batch[i] of samples from X
    Model recommends the best actions as the action with the highest reward probability to each simulated sample in batch[i] and collect corresponding simulated rewards
    Model priors are updated using information from recommended actions and returned rewards
```
Finally, we can visualize the results of the simulation. As defined in the ground truth: 'a2' was the action recommended the most.

In [6]:
smab_simulator.run()

/home/runner/work/pybandits/pybandits/pybandits/simulator.py:336: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self._results = pd.concat((self._results, batch_results), ignore_index=True)
2026-01-14 00:12:46.027 | INFO     | pybandits.simulator:_print_results:587 - Simulation results (first 10 observations):



2026-01-14 00:12:46.035 | INFO     | pybandits.simulator:_print_results:588 - Count of actions selected by the bandit: 



2026-01-14 00:12:46.038 | INFO     | pybandits.simulator:_print_results:589 - Observed proportion of positive rewards for each action:



Loading BokehJS ...

Furthermore, we can examine the number of times each action was selected and the proportion of positive rewards for each action.

In [7]:
smab_simulator.selected_actions_count

action,a1,a2,a3,cum_a1,cum_a2,cum_a3
batch,,,,,,
0,31,30,39,31,30,39
1,0,100,0,31,130,39
2,0,100,0,31,230,39
3,0,100,0,31,330,39
4,0,100,0,31,430,39
5,0,100,0,31,530,39
6,0,100,0,31,630,39
7,0,100,0,31,730,39
8,0,100,0,31,830,39


In [8]:
smab_simulator.positive_reward_proportion

,proportion
action,
a1,0.064516
a2,0.791398
a3,0.0
